<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/10%20-%20Graph%20Neural%20Networks%20and%20MLIPs/10b_chemprop.ipynb)

# 10b · chemprop：一行命令训练图神经网络

> 对应 ai4chem `03 - Intro to Deep Learning` 的 `03_gnn_simple_example.ipynb`。原课程用 ESOL 数据和 chemprop 的随机划分；这里用同一份 AqSolDB，并让 chemprop 用**和 07a、10a 完全相同的测试集**，结果可以直接比较。

1. chemprop 是什么？D-MPNN 和 10a 的 GNN 有什么不同
2. 准备数据和划分文件
3. `chemprop train`：训练
4. `chemprop predict`：预测，并和所有模型比较
5. 用训练好的模型预测有机半导体分子

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "10 - Graph Neural Networks and MLIPs"))
    get_ipython().run_line_magic("pip", "install -q rdkit chemprop")


In [1]:
try:
    import chemprop
except ImportError:
    %pip install -q chemprop
    import chemprop

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split

RDLogger.DisableLog("rdApp.*")
os.environ["PYTHONIOENCODING"] = "utf-8"   # Windows 上防止 chemprop 打印进度条时出现编码错误；其他系统无影响
print("chemprop 版本:", chemprop.__version__)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
chemprop 版本: 1.6.1


<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


💡 Colab 上第一次安装 chemprop 大约需要 1 分钟。本 notebook 基于 chemprop 2.x；1.x 的命令行写法不同（例如 `chemprop_train`），网上的旧教程要注意区分。

---
## 1. chemprop 是什么？

📘 [chemprop](https://github.com/chemprop/chemprop) 是 MIT 开发的分子性质预测工具，核心是**有向消息传递神经网络**（D-MPNN）。它在很多基准测试里表现很好，也被用于真实的发现工作，例如用它筛选出了新的抗生素 halicin（Stokes et al., *Cell* 2020）。

D-MPNN 和 10a 我们自己写的 GNN 几乎一样，只有一个关键区别：**消息挂在"有向键"上，而不是原子上**。

| | 10a 的 GNN | chemprop（D-MPNN） |
|---|---|---|
| 隐藏状态在哪里 | 每个原子一个 $h_i$ | 每根键的每个方向一个 $h_{i \to j}$ |
| 消息传递 | 原子 $i$ 收集所有邻居的消息 | 键 $i \to j$ 收集所有**指向 $i$ 的键**的消息，但**排除**来自 $j$ 的那根 |
| 读出 | 原子向量求和 | 先把键向量汇总回原子，再对原子求和（默认再除以常数 100） |

🤔 为什么要排除"来自 $j$ 的那根"？

📘 在 10a 的 GNN 里，原子 $i$ 第一轮把信息传给邻居 $j$，第二轮 $j$ 又把这条信息原封不动地传回给 $i$。这叫**来回震荡**（tottering），浪费了层数，信息"原地打转"。D-MPNN 规定消息不能沿着来的那根键立刻走回去，每一层都能真正向外看一步。

💡 chemprop 还帮我们做好了很多 07a、10a 里手写的杂事：目标值标准化、学习率预热和衰减、早停（保存验证集上最好的模型）、批处理等等。

---
## 2. 准备数据

In [2]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 solubility.csv 即可
DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(DATA_FILE)
print("读取成功！一共", len(df), "个分子")

读取成功！一共 9482 个分子


📘 chemprop 只需要一个 CSV 文件：一列 SMILES，一列（或几列）目标值。分子特征它自己从 SMILES 算。

先去掉 RDKit 无法解析的 2 个分子，保持和前面完全一样的 9480 个分子和同样的行顺序：

In [3]:
df = df[df["SMILES"].apply(Chem.MolFromSmiles).notna()].reset_index(drop=True)

os.makedirs("chemprop_data", exist_ok=True)
DATA_CSV = "chemprop_data/solubility_clean.csv"
df[["SMILES", "Solubility"]].to_csv(DATA_CSV, index=False)
print(len(df), "个分子已写入", DATA_CSV)
pd.read_csv(DATA_CSV).head(3)

9480 个分子已写入 chemprop_data/solubility_clean.csv


,SMILES,Solubility
0,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127
1,O=C1Nc2cccc3cccc1c23,-3.254767
2,Clc1ccc(C=O)cc1,-2.177078


### 2.1 划分文件：用和 07a、10a 一样的测试集

📘 chemprop 默认会自己随机划分 80/10/10。为了公平比较，我们用 `--splits-file` 告诉它每个分子属于哪个集合：一个 JSON 文件，里面写着训练集、验证集、测试集的**行号**。

In [ ]:
idx = np.arange(len(df))
idx_trval, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_valid = train_test_split(idx_trval, test_size=0.1, random_state=42)

SPLITS_JSON = "chemprop_data/splits.json"
with open(SPLITS_JSON, "w") as f:
    json.dump([{"train": idx_train.tolist(), "val": idx_valid.tolist(), "test": idx_test.tolist()}], f)
print("训练:", len(idx_train), " 验证:", len(idx_valid), " 测试:", len(idx_test))

# 测试集单独存一份，后面用 chemprop predict 预测
TEST_CSV = "chemprop_data/test.csv"
df.iloc[idx_test][["SMILES", "Solubility"]].to_csv(TEST_CSV, index=False)

---
## 3. 训练：`chemprop train`

📘 `!` 开头的一行不是 Python，而是交给系统命令行执行的命令。`{DATA_CSV}` 这种写法会被替换成 Python 变量的值。

参数的意思：

| 参数 | 含义 |
|---|---|
| `-i` | 数据文件 |
| `-s` / `--target-columns` | SMILES 列 / 目标值列的名字 |
| `--task-type` | `regression`（回归）或 `classification`（分类）等 |
| `--splits-file` | 自定义划分；不给的话默认随机 80/10/10 |
| `--epochs` | 训练轮数（默认 50） |
| `--metrics` | 要报告的指标，第一个用于选最好的模型 |
| `-o` | 结果保存到哪个文件夹 |

chemprop 训练时会打印几万行进度条。命令末尾的 `> 文件名 2>&1` 把这些输出都写进一个日志文件，notebook 里就清爽了。训练完再用下面的 `show_log` 只挑出我们关心的几行。

▶️ 训练 30 轮（CPU 上大约 2–3 分钟）。运行期间格子左边显示 `[*]`，没有任何输出，耐心等它变成数字。

In [ ]:
def show_log(path, keyword="test/"):
    """打印日志里包含 keyword 的行；一行都没有（多半是出错了）就打印最后 20 行，方便排查"""
    lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
    hits = [line for line in lines if keyword in line]
    print("\n".join(hits) if hits else "\n".join(lines[-20:]))

In [ ]:
RUN_DIR = "chemprop_runs/solubility"
TRAIN_LOG = "chemprop_data/train.log"
!chemprop train -i {DATA_CSV} -s SMILES --target-columns Solubility --task-type regression --splits-file {SPLITS_JSON} --epochs 30 --pytorch-seed 0 --metrics rmse mae r2 -o {RUN_DIR} > {TRAIN_LOG} 2>&1
show_log(TRAIN_LOG)

💡 想看完整的训练过程，可以直接打开 `chemprop_data/train.log`。里面如果有大量 `WARNING: not removing hydrogen atom without neighbors`，可以忽略：那是数据集里的 H⁻、H₂ 一类特殊物种。

📘 训练结束后，`RUN_DIR` 里有：

- `model_0/best.pt`：验证集上最好的模型，预测时用它；
- `model_0/test_predictions.csv`：测试集的预测值；
- `config.toml`：这次训练用到的全部参数（包括所有我们没写、用了默认值的参数），方便以后复现。

In [ ]:
for root, dirs, files in os.walk(RUN_DIR):
    depth = root[len(RUN_DIR):].count(os.sep)
    if depth <= 1:
        for name in files:
            print(os.path.join(root, name))

In [ ]:
print(open(os.path.join(RUN_DIR, "config.toml")).read()[:1200])

🤔 在 `config.toml` 里找一找：消息传递有几层（`depth`）？隐藏层多大（`message-hidden-dim`）？学习率是怎么设置的？和 10a 我们自己的 GNN（3 层，64 维，固定学习率 1e-3）比较一下。

📘 chemprop 默认 3 层消息传递、300 维隐藏状态，学习率先从 1e-4 **预热**到 1e-3，再逐渐降回 1e-4。07a 第 7 节提到过的学习率调度，就是这个。

---
## 4. 预测：`chemprop predict`

📘 用训练好的模型预测新分子，只需要一个有 SMILES 列的 CSV。我们先拿测试集来预测，这样能和真实值比较。

In [ ]:
MODEL = os.path.join(RUN_DIR, "model_0", "best.pt")
TEST_PRED_CSV = "chemprop_data/test_predictions.csv"
!chemprop predict -i {TEST_CSV} -s SMILES --model-paths {MODEL} -o {TEST_PRED_CSV} > chemprop_data/predict.log 2>&1
show_log("chemprop_data/predict.log", keyword="saved to")

In [ ]:
test_pred = pd.read_csv(TEST_PRED_CSV)
test_pred.head(3)

📘 预测结果和输入文件的行顺序一样，预测值放在与训练目标同名的列（`Solubility`）里。真实值要从我们自己的 `df` 里取。

In [ ]:
y_true = df["Solubility"].values[idx_test]
y_pred = test_pred["Solubility"].values

chemprop_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
print(f"chemprop 测试 RMSE = {chemprop_rmse:.2f},  R² = {r2:.2f}")

plt.figure(figsize=(4.5, 4.5))
plt.scatter(y_true, y_pred, s=5, alpha=0.4)
plt.plot([-13, 3], [-13, 3], "k--", lw=1)
plt.xlabel("measured logS"); plt.ylabel("predicted logS"); plt.title("chemprop on test set")
plt.tight_layout(); plt.show()

### 4.1 第二、三部分所有模型的比较

所有数字都在**同一个测试集**（1896 个分子）上：

In [ ]:
results = pd.DataFrame([
    ("always predict the mean", "—", 2.42),
    ("linear regression (02a)", "17 descriptors", 1.64),
    ("kNN, scaled (02a)", "17 descriptors", 1.24),
    ("random forest (02a)", "17 descriptors", 1.17),
    ("MLP (03a)", "17 descriptors", 1.19),
    ("random forest (02a)", "descriptors + count FP", 1.11),
    ("our GNN (03b)", "atoms + bonds", 1.04),
    ("chemprop D-MPNN (03c)", "atoms + bonds", round(chemprop_rmse, 2)),
], columns=["model", "input", "test RMSE"])

fig, ax = plt.subplots(figsize=(7, 3.5))
labels = [f"{m}\n[{i}]" for m, i in zip(results["model"], results["input"])]
colors = ["gray"] + ["tab:blue"] * 5 + ["tab:green"] * 2
ax.barh(labels, results["test RMSE"], color=colors)
ax.invert_yaxis(); ax.set_xlabel("test RMSE (logS)"); ax.set_title("Same test set, different models")
for k, v in enumerate(results["test RMSE"]):
    ax.text(v + 0.02, k, f"{v:.2f}", va="center")
plt.tight_layout(); plt.show()

📘 读这张图的几个要点：

- **最大的一步**是从"什么都不学"（2.42）到"随便一个模型"（1.64），然后到非线性模型（约 1.2）。
- 在**手工特征**上，随机森林、kNN、MLP 都挤在 1.17–1.24 附近：07a 说的"瓶颈在特征"。
- 两个**图神经网络**（绿色）只看原子和键，却都做到了 1.04–1.05，是所有模型里最好的。chemprop 和我们 50 行代码的 GNN 差不多：**思路对了，具体实现的差别没那么大**。
- 但也要实事求是：从最好的随机森林（1.11）到 GNN（约 1.05），提升只有 0.06 左右，而训练时间多了几十倍。溶解度测量本身的误差通常就有 0.5 个 log 单位以上（不同实验室测同一个分子，结果常常差这么多），模型离这个"天花板"已经不太远了。

💡 想进一步提高，常见的思路是：更多的训练轮数和**集成**（`--ensemble-size 5`，训练 5 个模型取平均，chemprop 还会给出它们的分歧作为不确定度）、加入额外的分子特征（`--molecule-featurizers`）、调超参数（`chemprop hpopt`）。

---
## 5. 预测有机半导体分子

📘 和 02c 第 5 节一样，看看模型对有机电子学里常见的分子怎么说。

In [ ]:
candidates = {
    "thiophene": "c1ccsc1",
    "EDOT": "C1COc2cscc2O1",
    "3-hexylthiophene": "CCCCCCc1ccsc1",
    "carbazole": "c1ccc2c(c1)[nH]c1ccccc12",
    "triphenylamine": "c1ccc(cc1)N(c1ccccc1)c1ccccc1",
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "BTBT": "c1ccc2c(c1)sc1c3ccccc3sc21",
    "C8-BTBT": "CCCCCCCCc1ccc2c(c1)sc1c3ccc(CCCCCCCC)cc3sc21",
}
OSC_CSV = "chemprop_data/osc.csv"
OSC_PRED_CSV = "chemprop_data/osc_predictions.csv"
pd.DataFrame({"name": list(candidates), "SMILES": list(candidates.values())}).to_csv(OSC_CSV, index=False)
!chemprop predict -i {OSC_CSV} -s SMILES --model-paths {MODEL} -o {OSC_PRED_CSV} > chemprop_data/predict.log 2>&1
show_log("chemprop_data/predict.log", keyword="saved to")

In [ ]:
osc = pd.read_csv(OSC_PRED_CSV).rename(columns={"Solubility": "predicted logS"})

canon_to_logS = dict(zip(df["SMILES"].apply(Chem.CanonSmiles), df["Solubility"]))
osc["measured logS"] = [canon_to_logS.get(Chem.CanonSmiles(s), np.nan) for s in osc["SMILES"]]
osc[["name", "predicted logS", "measured logS"]].round(2)

🤔 看看表格：

1. 哪些分子在数据集里有实验值？模型对它们预测得怎么样？
2. 噻吩 → 3-己基噻吩，C8-BTBT 和 BTBT 相比，烷基链让**水**溶解度变好还是变差？这和 02c 里的讨论一致吗？
3. C8-BTBT 预测出来的 logS 是多少？换算成浓度大约是每升多少摩尔？你相信这个数吗？（提示：想想数据集里有没有这么难溶的分子，以及 02b 讲的适用域。）

📘 参考（不同电脑上结果可能在小数点后第二位有差别）：噻吩预测约 -1.4，实验值 -1.45，非常准；咔唑预测约 -4.5，实验值 -5.21，偏高了 0.7 左右。接上烷基链后，3-己基噻吩比噻吩低了约 3.5，C8-BTBT 比 BTBT 低了约 1.1：**在水里**，侧链只会让分子更难溶，和 02c 的讨论一致。C8-BTBT 预测约 -7.5，也就是约 $3 \times 10^{-8}$ mol/L。数据集里比这还难溶的分子很少，而且几乎没有长烷基链的稠环分子，这个数只能当作"非常难溶"的定性判断。

### 🏋️ 练习 10b-1：用 chemprop 做分类

把任务改成 02c 的"logS < -4 是否难溶"：

1. 在 `df` 里新建一列 `insoluble = (Solubility < -4).astype(int)`，和 SMILES 一起存成新的 CSV；
2. 用 `--task-type classification`、`--target-columns insoluble`、`--metrics roc prc accuracy`，同一个 `--splits-file`，训练 30 轮；
3. 测试集 ROC-AUC 是多少？和 02c 随机森林的 0.953 比较。

### 🏋️ 练习 10b-2：scaffold split

02b 第 6 节比较过随机划分和 scaffold split。chemprop 自带 scaffold split：**去掉** `--splits-file`，改用 `--split-type SCAFFOLD_BALANCED --split-sizes 0.8 0.1 0.1`，训练 30 轮。测试 RMSE 比随机划分差多少？和 02b 的结论（随机森林 1.17 → 1.20）比，GNN 受的影响更大还是更小？

⚠️ 这时测试集已经不是同一批分子了，只能大致比较。

In [ ]:
# 在这里写你的代码

---
## 小结：第三部分结束

| 学到了 | 代码 / 要点 |
|---|---|
| 训练 | `chemprop train -i data.csv -s SMILES --target-columns y --task-type regression -o out/` |
| 自定义划分 | `--splits-file`（JSON，写训练/验证/测试的行号） |
| 预测 | `chemprop predict -i new.csv -s SMILES --model-paths out/model_0/best.pt -o preds.csv` |
| D-MPNN | 消息挂在有向键上，不沿来路返回，避免来回震荡 |
| 复现 | `config.toml` 记录了全部参数 |

第三部分的三个 notebook：

- **07a**：神经网络的基本零件（张量、自动求导、梯度下降、`nn.Module`、训练循环、正则化）。在 17 个描述符上，MLP 并不比随机森林强。
- **10a**：把分子当成图，自己写了消息传递。只看原子和键的 GNN 超过了所有描述符模型，而且对并苯系列的外推更合理。
- **10b**：用 chemprop 一行命令得到同样水平的结果。

**第四部分（无监督学习）**：到目前为止每个分子都有标签（logS）。如果手里只有一大堆分子结构、没有任何测量值，机器学习还能做什么？降维、聚类、化学空间可视化。